## COL funtional analysis 012025 & 032024 reads

**NOTE: was unable to get Humann4 to work after LOTS of troubleshooting and communicating with developers, sifting through biobakery forum, etc. Was able to finally get humann3 to work with specific software versions and metaphlan databases, so scroll down to get to scripts that actually worked**

### Humann4 (HMP Unified Metabolic Analysis Network)
https://github.com/biobakery/humann#configuration \
https://github.com/biobakery/biobakery/wiki/humann4#12-installation
https://docs.google.com/document/d/1rCx5JkuO7wCKWrL8_-UJx_FkopJAfcDFtZktgPspak0/edit?tab=t.0

In [ ]:
salloc -p cpu -t 8:00:00 -c 4
#INSTALLATION
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4
#set conda channel priority, biobakery should be top
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge
conda config --add channels biobakery
conda install -c biobakery humann

was unable to download the databases the way they recommend with humann_databases --download so I took a different approach

In [ ]:
#THIS WORKS FOR DATABASE DOWNLOAD
mkdir utility_mapping
curl -L https://g-227ca.190ebd.75bc.data.globus.org/humann_data/full_mapping_v4_alpha.tar.gz --output utility_mapping.tar.gz
tar -xvzf utility_mapping.tar.gz
humann_config --update database_folders utility_mapping /scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4/utility_mapping
# wrote a script for downloading the other databases too

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=25G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-humann4db_update-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4

DBPATH="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4"
CHOCO="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4/chocophlan"
UNIREF="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4/uniref"

cd $CHOCO/

#update CHOCOPHLAN database
curl -L http://huttenhower.sph.harvard.edu/humann_data/chocophlan/chocophlan.v4_alpha.tar.gz --output chocophlan.tar.gz
    echo "chocophlan database downloaded"
cd $CHOCO   
tar -xvzf chocophlan.tar.gz
    echo "chocophlan database unzipped"
humann_config --update database_folders nucleotide $CHOCO
    echo "config file updated for chocophlan"

cd $UNIREF/
#update UNIREF database
wget http://huttenhower.sph.harvard.edu/humann_data/uniprot/uniref_ec_filtered/uniref90_annotated_v4_alpha_ec_filtered.tar.gz
    echo "uniref database downloaded"
cd $UNIREF
tar -xvzf uniref90_annotated_v4_alpha_ec_filtered.tar.gz
      echo "uniref90 database unzipped" 
humann_config --update database_folders protein $UNIREF
    echo "config file updated for uniref90"

#cd $DBPATH
#metaphlan --install --index mpa_vOct22_CHOCOPhlAnSGB_202403 
	
conda deactivate

# JOB-ID: 56149791
# bash script file name: humann4db_update.sh

In [ ]:
#install the correct metaphlan markers that work with this humann version
metaphlan --install --index mpa_vOct22_CHOCOPhlAnSGB_202403

doesn't need to be in a script - can just be in a tmux session with salloc! Althought it was nice to have the job running for the tar unzip step for chocophlan. curl only worked for chocophlan and utility mapping, wget only worked for uniref. don't ask me why

**Make sure to move the .tar.gz files out of the chocophlan, etc directories**

**This analysis is following lots of playing around with different kraken2 databases, and then filtering out reads based on kraken ID (PrackenDB).**

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-dlab-032024-humann4-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4

# Set parameters
SAMPLENAME="dlab"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated"
mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4_output/${SAMPLENAME}"
mkdir -p $OUTDIR

#work from scratch4 directory (submit script from there) because the temp files are very big

#concatenate F and R of kraken2 filtered reads
#I don't think kraken2 rezipped the fastqs after filtering so let's see if this helps (humann couldnt unzip)
while IFS= read -r SAMPLEID; do
cat $READS/"${SAMPLEID}"_R1_kraken_filtered.fastq.gz $READS/"${SAMPLEID}"_R2_kraken_filtered.fastq.gz > $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq
 if [ $? -eq 0 ]; then
        echo "concatenation successful for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"

cd $OUTDIR

while IFS= read -r SAMPLEID; do
humann --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output "${SAMPLEID}"_humann --threads $SLURM_CPUS_ON_NODE
 if [ $? -eq 0 ]; then
        echo "humann4 profile completed for sample: $SAMPLEID"
    else
        echo "humann4 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate

# JOB-ID: 56221481 and 56230839
# bash script file name: COL/bash_scripts/COL_humann4_dlab_032024.sh

In [ ]:
#ERROR: The MetaPhlAn taxonomic profile provided does not contain the database version vOct22_CHOCOPhlAnSGB_202403 in any of its header lines. 
#need to install the correct metaphlan4 file (if it is there already, delete and redownload)
metaphlan --install --index mpa_vOct22_CHOCOPhlAnSGB_202403

I still keep getting this error
ERROR: The MetaPhlAn taxonomic profile provided does not contain the database version vOct22_CHOCOPhlAnSGB_202403 in any of its header lines.

this resource: https://forum.biobakery.org/t/metaphlan-4-humann-4-compatibility/8523/8 recommends specifying the paths for each database so I will try that in the above script. Making those changes below.

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-dlab-032024-humann4-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4

# Set parameters
SAMPLENAME="dlab"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated"
#mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4_output/${SAMPLENAME}"
#mkdir -p $OUTDIR
DB="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann4"
METAPHLAN="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann4/lib/python3.12/site-packages/metaphlan/metaphlan_databases"

#work from scratch4 directory (submit script from there) because the temp files are very big

#concatenate F and R of kraken2 filtered reads
#kraken2 doesnt rezip the fastqs after filtering (need to change the output file names in the filter script)

cd $OUTDIR

#run humann4
while IFS= read -r SAMPLEID; do
humann -r --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output "${SAMPLEID}"_humann --threads $SLURM_CPUS_ON_NODE \
--protein-database $DB/uniref --nucleotide-database $DB/chocophlan --metaphlan-options "-t rel_ab_w_read_stats \
--bowtie2db $METAPHLAN/vOct22 --index mpa_vOct22_CHOCOPhlAnSGB_202403"
 if [ $? -eq 0 ]; then
        echo "humann4 profile completed for sample: $SAMPLEID"
    else
        echo "humann4 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate

# JOB-ID: 56231759
# bash script file name: COL/bash_scripts/COL_humann4_dlab_032024.sh

### didn't work. pivoting to humann3

In [ ]:
##INSTALLATION
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3
conda install -c biobakery humann==3.9

#downloading databases
curl -L https://huttenhower.sph.harvard.edu/humann_data/full_mapping_v201901b.tar.gz --output utility_mapping.tar.gz
tar -xvzf utility_mapping.tar.gz 
humann_config --update database_folders utility_mapping /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases/utility_mapping
mv utility_mapping.tar.gz ../

wget http://huttenhower.sph.harvard.edu/humann_data/chocophlan/full_chocophlan.v201901_v31.tar.gz
tar -xvzf full_chocophlan.v201901_v31.tar.gz
humann_config --update database_folders nucleotide /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases/chocophlan
mv full_chocophlan.v201901_v31.tar.gz ../

I wasn't able to download uniref databases using curl or wget. But I was able to manually download to my local downloads folder with the links: https://huttenhower.sph.harvard.edu/humann_data/uniprot/uniref_ec_filtered/uniref50_ec_filtered_201901b_subset.tar.gz and https://huttenhower.sph.harvard.edu/humann_data/uniprot/uniref_annotated/uniref50_annotated_v201901b_full.tar.gz 

Then I used globus to transfer these to Unity /uniref folder.

In [ ]:
#using the full (non-ec filtered) uniref 50 database
tar -xvzf uniref50_annotated_v201901b_full.tar.gz
humann_config --update database_folders protein /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases/uniref
mv uniref50_annotated_v201901b_full.tar.gz ../

In [ ]:
Kept getting: ModuleNotFoundError: No module named 'distutils'
#install the correct metaphlan version
pip install setuptools #this did the trick to get MetaPhlAn version 4.0.6 (1 Mar 2023) - hopefully this is compatible with humann3
conda install metaphlan -c bioconda

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-dlab-032024-humann3-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3

# Set parameters
SAMPLENAME="dlab"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated"
#mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann3_output/${SAMPLENAME}"
mkdir -p $OUTDIR
DB="/project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases"

#work from scratch4 directory (submit script from there) because the temp files are very big

#concatenate F and R of kraken2 filtered reads
#kraken2 doesnt rezip the fastqs after filtering (need to change the output file names in the filter script)

cd $OUTDIR

#run humann4
while IFS= read -r SAMPLEID; do
humann --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output $OUTDIR/"${SAMPLEID}"_humann --threads $SLURM_CPUS_PER_TASK
 if [ $? -eq 0 ]; then
        echo "humann3 profile completed for sample: $SAMPLEID"
    else
        echo "humann3 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate

# JOB-ID: 56395250
# bash script file name: COL/bash_scripts/COL_humann3_dlab_032024.sh

it had an issue first with CRITICAL ERROR: Can not call software version for metaphlan so I installed setuptools as I've noted above.

In [ ]:
#Then it said: ERROR: The MetaPhlAn taxonomic profile provided was not generated with the database version v3 or vJun23 . Please update your version of MetaPhlAn to at least v3.0 or if you are using MetaPhlAn v4 please use the database vJun23.
#so I did this
cd /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases
metaphlan --install --index mpa_vJun23_CHOCOPhlAnSGB_202403
#made a directory for the vJan25 to keep separate, then restarted the job
#for some reason these files disappeared?! maybe this happened after downgrading metaphlan?

In [ ]:
#Then it said: The MetaPhlAn taxonomic profile provided was not generated with the database version v3 or vJun23 . Please update your version of MetaPhlAn to at least v3.0 or if you are using MetaPhlAn v4 please use the database vJun23.
#thought to downgrade metaphlan but then I couldnt download indexes, so I went back to metaphlan 4.0 and re downloaded indexes
cd /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases
metaphlan --install --index mpa_vJun23_CHOCOPhlAnSGB_202403

Trying a different approach with being more specific in the humann3 command

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-dlab-032024-humann3-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3

# Set parameters
SAMPLENAME="dlab"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated"
#mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann3_output/${SAMPLENAME}"
mkdir -p $OUTDIR
DB="/project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases"
METAPHLAN="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases/"

#concatenate F and R of kraken2 filtered reads
#kraken2 doesnt rezip the fastqs after filtering (need to change the output file names in the filter script)

cd $OUTDIR

#run humann3
while IFS= read -r SAMPLEID; do
humann --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output "${SAMPLEID}"_humann --threads $SLURM_CPUS_PER_TASK \
--protein-database $DB/uniref --nucleotide-database $DB/chocophlan --metaphlan $METAPHLAN --metaphlan-options "-t rel_ab_w_read_stats"
 if [ $? -eq 0 ]; then
        echo "humann3 profile completed for sample: $SAMPLEID"
    else
        echo "humann3 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"
#add -r for resuming a run

conda deactivate

# JOB-ID:  56460097
# bash script file name: COL/bash_scripts/COL_humann3_dlab_032024.sh

downloaded, changed the header to match whats in the metaphlan database: mpa_vJun23_CHOCOPhlAnSGB_202403 and then put in utility mapping directory
https://github.com/biobakery/MetaPhlAn/blob/master/metaphlan/utils/mpa_vJun23_CHOCOPhlAnSGB_202307_SGB2GTDB.tsv

I still get this. ERROR: The MetaPhlAn taxonomic profile provided was not generated with the database version v3 or vJun23 . Please update your version of MetaPhlAn to at least v3.0 or if you are using MetaPhlAn v4 please use the database vJun23.

Also this warning. WARNING: Can not call software version for bowtie2

The current metaphlan version is 4.0.6 but from https://forum.biobakery.org/t/humann-3-9-conda-install-will-not-work-please-update-your-version-of-metaphlan/8484/4 it looks like Metaphlan 4.1.1 is the way to go. I moved the vJun23 database out of metaphlan before re-installing. There is incompatibility here so I'm going to just try from the beginning by removing humann3, creating an env with metaphlan 4.1.1. and then installing humann3 AGAIN. T

**This didn't work, now will try metaphlan3**

In [ ]:
conda install bioconda::metaphlan==3.1.0
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge
conda config --add channels biobakery
conda install -c biobakery humann==3.9
pip install setuptools
metaphlan --version
#MetaPhlAn version 3.1.0 (25 Jul 2022)
humann --version
#humann v3.9

copied metaphlan vJun23 database from /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases/metaphlan/ to /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-dlab-032024-humann3-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3

# Set parameters
SAMPLENAME="dlab"
SAMPLELIST="032024_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated"
#mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann3_output/${SAMPLENAME}"
mkdir -p $OUTDIR
DB="/project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases"
METAPHLAN="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases"

#concatenate F and R of kraken2 filtered reads
#kraken2 doesnt rezip the fastqs after filtering (need to change the output file names in the filter script)

cd $OUTDIR

#run humann3
while IFS= read -r SAMPLEID; do
humann --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output "${SAMPLEID}"_humann --threads $SLURM_CPUS_PER_TASK \
--protein-database $DB/uniref --nucleotide-database $DB/chocophlan --metaphlan $METAPHLAN --metaphlan-options "-t rel_ab_w_read_stats"
 if [ $? -eq 0 ]; then
        echo "humann3 profile completed for sample: $SAMPLEID"
    else
        echo "humann3 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"
#add -r for resuming a run

conda deactivate

# JOB-ID: 56682561 
# bash script file name: COL/bash_scripts/COL_humann3_dlab_032024.sh

THIS SCRIPT WORKED!! Needed 12 extra hours, so keep that in mind for the rest of the scripts. and the last sample didn't finish.

**Ok going back to the filtered reads that were then pushed through PlusPF db for final taxonomic classification using the below scripts. 1. concatentate reads 2. run through humann3**

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 72:00:00  # Job time limit
#SBATCH --qos=long #if job takes longer than 48 hrs
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-mcav-032024-humann3-%j.out  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3

# Set parameters
SAMPLENAME="mcav"
DATE="032024"
SAMPLELIST="${DATE}_${SAMPLENAME}_sampleids.txt" 
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/${DATE}"
CAT_READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/kraken2_filtered_reads/concatentated_reads"
mkdir -p $CAT_READS
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann3_output/${SAMPLENAME}"
mkdir -p $OUTDIR
DB="/project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/humann3_databases"
METAPHLAN="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/humann3/lib/python3.13/site-packages/metaphlan/metaphlan_databases"

#concatenate F and R of kraken2 filtered reads
while IFS= read -r SAMPLEID; do
cat $READS/"${SAMPLEID}"_R1_kraken_filtered.fastq $READS/"${SAMPLEID}"_R2_kraken_filtered.fastq > $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq
 if [ $? -eq 0 ]; then
        echo "concatenation successful for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"


cd $OUTDIR

#run humann3
while IFS= read -r SAMPLEID; do
humann --input $CAT_READS/"${SAMPLEID}"_comb_kraken_filtered.fastq --output "${SAMPLEID}"_humann --threads $SLURM_CPUS_PER_TASK \
--protein-database $DB/uniref --nucleotide-database $DB/chocophlan --metaphlan $METAPHLAN --metaphlan-options "-t rel_ab_w_read_stats"
 if [ $? -eq 0 ]; then
        echo "humann3 profile completed for sample: $SAMPLEID"
    else
        echo "humann3 encountered an error for sample: $SAMPLEID"
        exit 1
    fi 
done < "$LISTPATH/${SAMPLELIST}"
#add -r for resuming a run

conda deactivate

# JOB-ID: 57405476
# bash script file name: COL/bash_scripts/COL_humann3_mcav_032024.sh

job IDs for the others: \
012025 mcav: 57407255 and 57464852 \
032024 ofav: 57465304 (using new sampleids.txt) \
012025 ofav: 57465305 \
032024 dlab: 57465307 \
012025 dlab: 57465308 \
012025 pstr: \
032024 pstr: 57550282 (using new sampleids.txt)

#### Processing humann3 output

copied all genefamilies.tsv into /scratch4/workspace/nikea_ulrich_uml_edu-col_data/humann3_output/combined_genefamilies

*pstr samples haven't completed running yet when I did this initial processing (5/11/26)*

In [ ]:
humann_join_tables -i ./ -o mcav_dlab_ofav_genefamilies.tsv 

In [ ]:
humann_renorm_table --input mcav_dlab_ofav_genefamilies.tsv --output mcav_dlab_ofav_genefamilies_cpm.tsv --units cpm --update-snames

In [ ]:
humann_regroup_table --input mcav_dlab_ofav_genefamilies_cpm.tsv --output mcav_dlab_ofav_genefamilies_rxn_cpm.tsv --groups uniref50_rxn

In [ ]:
humann_rename_table --input mcav_dlab_ofav_genefamilies_rxn_cpm.tsv --output mcav_dlab_ofav_genefamilies_rxn_cpm_named.tsv --names metacyc-rxn

**further data analysis in COL_functional_viz.ipnyb**

Did above including 032024 PSTR samples (file: all_but_0125pstr_genefamilies.tsv). Will include 012025 PSTRs asap.